In [0]:
%sql
--setting up the ingestion layer and practicing reading from a volume in databricks

--read_files can detect file format automatically and infer a unified schema across all files
/*
SELECT * 
FROM read_files(
    '/Volumes/connections/default/raw_data/GettyAAT.json',
    format => 'json'
)
LIMIT 1;
*/
--testing to see how read_files() handles the json
--appears to be a nested array, which is what we expected to see 


In [0]:
%sql
--creating a table and loading the data

CREATE TABLE IF NOT EXISTS connections.default.getty_aat
AS
SELECT *
FROM read_files(
    '/Volumes/connections/default/raw_data/GettyAAT.json',
    format => 'json'
);

SELECT * FROM connections.default.getty_aat LIMIT 10;

--inspecting the schema
DESCRIBE connections.default.getty_aat;
--Success! Databricks correctly inferred the nested structure for the JSON


In [0]:
%sql
SELECT
`_label`,
broader
FROM connections.default.getty_aat;

/* 
rhyta
array
0: {"_label": "drinking vessels", "id": "http://vocab.getty.edu/aat/300194567", "type": "Type"}*/
--returns the label rhyta and the braoder concept of drinking vessels 

--try explode on the nested array to see each broader concept as its own row
SELECT 
_label,
explode(broader) as broader_concept
FROM connections.default.getty_aat;
--can't tell if explode worked because rhyta only had one broader concept to begin with 

--trying again but filtering for broader concept > 1
SELECT
_label,
explode(broader) as broader_concept
FROM connections.default.getty_aat
WHERE size(broader) > 1
LIMIT 10;

--just realized I downloaded the individual JSON endpoint and not the full dataset
--this was a good practice run for ingestion but that's why only one record is persistently being returned